In [1]:
from pathlib import Path
import os
import numpy as np
import pandas as pd


data_set_path = (
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset.csv"
)

data_set_path = str(data_set_path)


def transform_skew_columns(data, skew_col_names):
    for col in skew_col_names:
        data[col] = np.log1p(data[col])


def transform_fattail_columns(data, fattail_col_names):
    for col in fattail_col_names:
        data[col] = np.arcsinh(data[col])


def fat_tail_scores(
    data_set_path,
    skew_thresh=4.0,
    **kwargs
):

    assert os.path.isfile(data_set_path), \
        f'"{data_set_path}" is not a valid path'

    churn_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    data_scores = churn_data.copy()

    if "is_churn" not in data_scores.columns:
        raise ValueError(
            '"is_churn" column not found.'
        )

    data_scores = data_scores.drop(
        columns=["is_churn"]
    )

    data_scores = data_scores.select_dtypes(
        include="number"
    )

    stat_path = data_set_path.replace(
        ".csv",
        "_summarystats.csv"
    )

    assert os.path.isfile(stat_path), \
        "You must run Listing 5.2 first to generate summary statistics."

    stats = pd.read_csv(
        stat_path,
        index_col=0
    )

    if "is_churn" in stats.index:
        stats = stats.drop(
            index="is_churn"
        )

    stats = stats.loc[
        stats.index.intersection(
            data_scores.columns
        )
    ]

    data_scores = data_scores[
        stats.index
    ]

    # Positive-only strongly skewed variables
    skewed_columns = (
        (stats["skew"] > skew_thresh)
        &
        (stats["min"] >= 0)
    )

    transform_skew_columns(
        data_scores,
        skewed_columns[
            skewed_columns
        ].index
    )

    # Strongly skewed variables that contain negatives
    fattail_columns = (
        (stats["skew"].abs() > skew_thresh)
        &
        (stats["min"] < 0)
    )

    transform_fattail_columns(
        data_scores,
        fattail_columns[
            fattail_columns
        ].index
    )

    # Recalculate scaling parameters after transforms
    mean_vals = data_scores.mean()
    std_vals = data_scores.std()

    # Remove zero-variance columns
    zero_std_cols = std_vals[
        std_vals == 0
    ].index

    if len(zero_std_cols) > 0:
        data_scores = data_scores.drop(
            columns=zero_std_cols
        )

        mean_vals = mean_vals.drop(
            index=zero_std_cols
        )

        std_vals = std_vals.drop(
            index=zero_std_cols
        )

        skewed_columns = skewed_columns.drop(
            index=zero_std_cols,
            errors="ignore"
        )

        fattail_columns = fattail_columns.drop(
            index=zero_std_cols,
            errors="ignore"
        )

    # Standardize
    data_scores = (
        data_scores - mean_vals
    ) / std_vals

    # Reattach target
    data_scores["is_churn"] = (
        churn_data["is_churn"]
    )

    # Save scores
    score_save_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    data_scores.to_csv(
        score_save_path,
        header=True
    )

    print(
        f"Saving results to:\n"
        f"{score_save_path}"
    )

    # Save transformation parameters
    param_df = pd.DataFrame({
        "skew_score": skewed_columns,
        "fattail_score": fattail_columns,
        "mean": mean_vals,
        "std": std_vals
    })

    param_save_path = data_set_path.replace(
        ".csv",
        "_score_params.csv"
    )

    param_df.to_csv(
        param_save_path,
        header=True
    )

    print(
        f"\nSaving params to:\n"
        f"{param_save_path}"
    )

    return data_scores, param_df


data_scores, score_params = fat_tail_scores(
    data_set_path=data_set_path,
    skew_thresh=4.0
)

display(data_scores.head())
display(score_params)

Saving results to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_scores.csv

Saving params to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_score_params.csv


,,like_per_month,account_tenure,is_churn
account_id,observation_date,,,
82,2020-02-09,0.516299,-1.235197,f
122,2020-02-09,-1.429966,-1.235197,f
184,2020-02-09,-1.135170,-1.235197,f
262,2020-02-09,-0.677778,-1.235197,f
291,2020-02-09,-0.830943,-1.235197,f


,skew_score,fattail_score,mean,std
like_per_month,True,False,3.980947,1.247389
account_tenure,False,False,61.239110,30.148318
